# Unit 09 - Trust Checks (Demo)

**Atoms served:** `U09-A7` (sanity checks), `U09-A8` (`SRM` - **no video**), `U09-A9` (A/A - **no video**), `U09-A10` (novelty over time)

**Estimated runtime:** ~25 seconds

**After this notebook you can:** run an `SRM` chi-square check, simulate an A/A false positive, and plot an effect that fades with novelty.

## Without code

1. `SRM` table: balanced split chi-square p-value should be high; 70/30 split should fail (p < 0.05).
2. A/A: among 1000 runs at alpha=0.05, about 50 should show p < 0.05 even with zero effect.
3. Novelty plot: week-1 lift should exceed week-4 lift.

`V27` shows balance instinct; `SRM` is the formal chi-square version - not named in any lecture.

## 1. The question

Before you trust a lift, ask: **did the experiment we ran match the experiment we designed?** Microsoft's MSN layout "won" partly because faster load times logged more users in the treatment arm - a failed ratio check.

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

Simulate assignment counts for a 50/50 test and a broken 70/30 split.

In [ ]:
n = 20000
balanced = pd.Series({'control': n//2, 'treatment': n//2})
broken = pd.Series({'control': int(0.30*n), 'treatment': int(0.70*n)})
print('Balanced counts:\n', balanced, '\nBroken counts:\n', broken)

## 4. The naive move

Skip balance checks and go straight to the treatment effect.

## 5. What actually happens

**`SRM` chi-square (`U09-A8`).** Compare observed split to designed 50/50.

In [ ]:
def srm_check(counts, expected_ratio=0.5):
    obs = counts.values
    chi2, p = stats.chisquare(obs, f_exp=[expected_ratio*obs.sum()]*2)
    return chi2, p

for label, counts in [('balanced', balanced), ('broken', broken)]:
    chi2, p = srm_check(counts)
    print(f"{label}: control={counts['control']}, treatment={counts['treatment']}, chi2={chi2:.2f}, p={p:.4f}")
    print('  FAIL SRM' if p < 0.05 else '  pass SRM')

The 70/30 split fails `SRM`. Treat the result as a data pipeline problem, not a win.

**A/A test (`U09-A9`) - not on video.** Same code path, zero true effect. Significant results still appear by chance.

In [ ]:
n_aa = 5000
n_sims = 1000
false_pos = 0
for _ in range(n_sims):
    y0 = np.random.binomial(1, 0.12, n_aa)
    y1 = np.random.binomial(1, 0.12, n_aa)  # identical arms
    _, p = stats.ttest_ind(y1, y0, equal_var=False)
    if p < 0.05:
        false_pos += 1
rate = false_pos / n_sims
print('A/A false positive rate at alpha=0.05:', round(rate, 3))
print('Expected ~0.05. Significance without an effect is cheap.')

About 5% of A/A runs "find" an effect - your own false positive rate on display.

**Novelty decay (`U09-A10`).** Plot weekly `ATE`; early weeks exaggerate change reactions (`V18` mentions novelty; detection method is Kohavi ch. 23).

In [ ]:
weeks = np.arange(1, 5)
# true stable effect 0.01, plus novelty bump decaying
novelty = np.array([0.03, 0.015, 0.005, 0.0])
observed = 0.01 + novelty
fig, ax = plt.subplots()
ax.plot(weeks, observed, 'o-')
ax.axhline(0.01, linestyle='--')
ax.set_xlabel('week')
ax.set_ylabel('observed ATE')
ax.set_title('Novelty inflates early weeks then fades')
plt.show()

## 6. What you do about it

- Run **sanity checks** from `V27` before inference (`U09-A7`).
- Formalise balance as **`SRM`** (`U09-A8`).
- Schedule **A/A** or shadow runs to calibrate false positives (`U09-A9`).
- Plot effects **over time** before calling a winner (`U09-A10`).

**When this matters less:** Tiny internal dogfood tests - still run `SRM`, skip A/A if traffic is zero.

---

**Takeaway:** A significant result on a broken experiment is Twyman bait. Trust checks are cheaper than wrong ship decisions.

**Back to the unit:** [V1](../V1/units/unit-09-validity-and-trust-checks/README.md) · [V2](../V2/units/unit-09-validity-and-trust-checks/README.md)